<a href="https://colab.research.google.com/github/zabdielVR/IA_with_collab/blob/main/Hands_On_Fine_Tuning_con_LoRA_(Soluci%C3%B3n).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FINE-TUNING CON LORA**

Una vez vista la masterclass ***Fine-tuning y Evaluación de Modelos***, se proporciona el siguiente ***Colab*** para ejecutar, en vivo, un fine-tuning real con LoRA sobre un modelo Llama ligero, y medir su mejora con una métrica objetiva.

A diferencia de los Temas anteriores, aquí no usamos Groq — Groq solo sirve para inferencia, no para entrenar modelos. Usamos **Hugging Face** (librerías `transformers` y `peft`) directamente sobre la GPU gratuita de Colab.

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura, añadiendo un nombre asociado al token para guardarlo dentro de una variable y usarlo dentro del notebook. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

In [11]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets

!pip install transformers peft accelerate trl --quiet

import torch
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("Sesión de Hugging Face iniciada correctamente.")

Sesión de Hugging Face iniciada correctamente.


### **CARGAR EL MODELO BASE**

Usamos un modelo Llama ligero (pocos parámetros) para que el fine-tuning corra en minutos sobre la GPU T4 gratuita de Colab, sin necesitar cuantización adicional.

In [12]:
# Cargar el modelo base de Llama y su tokenizer

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# variante oficial de Meta "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


### **ANTES DEL FINE-TUNING: LÍNEA BASE**

Antes de ajustar nada, probamos el modelo base con un prompt de ejemplo para tener un punto de comparación. El modelo aún no conoce el tono ni el formato que le vamos a enseñar.

In [13]:
# Definir una función para generar texto y probar el modelo base con un prompt de ejemplo

def generar_respuesta(modelo_a_usar, prompt, max_new_tokens=60):
    entrada = tokenizer(prompt, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)
    return texto_generado.split("\n")[0].strip()

prompt_prueba = "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente:"

respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

No, no puedes cambiar tu pedido.


### **PREPARAR LOS DATOS DE ENTRENAMIENTO**

El fine-tuning necesita ejemplos de entrada y salida que muestren el comportamiento que queremos enseñarle al modelo. Con pocos ejemplos (5 a 10) es suficiente para una demo — no es un dataset de producción.

In [14]:
# Definir una lista de ejemplos (entrada -> respuesta esperada) y convertirla en dataset

from datasets import Dataset

ejemplos = [
    {"texto": "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente: Sí, puedes "
     "solicitar el cambio dentro de la primera hora escribiendo a soporte@tienda.com."},
    {"texto": "Cliente: ¿Cuánto tarda el reembolso?\nAgente: El reembolso se refleja en un plazo de 5 a 7 días hábiles."},
    {"texto": "Cliente: ¿Tienen envío el mismo día?\nAgente: Sí, disponible en zonas seleccionadas si el pedido se confirma antes de las 12:00."},
    {"texto": "Cliente: ¿Puedo pagar en el momento de la entrega?\nAgente: Sí, aceptamos pago contra entrega en efectivo o tarjeta."},
    {"texto": "Cliente: ¿Cómo rastreo mi paquete?\nAgente: Puedes rastrearlo con el número de guía en la sección 'Mis pedidos' de tu cuenta."},
]

dataset = Dataset.from_list(ejemplos)
dataset

Dataset({
    features: ['texto'],
    num_rows: 5
})

## **REALIZAR FINE-TUNING**

### **CONFIGURAR Y APLICAR LORA**

LoRA agrega matrices pequeñas entrenables sin tocar los pesos originales del modelo — por eso es tan ligero comparado con un fine-tuning completo.

In [15]:
# Configurar LoRA (rango, alpha, módulos objetivo) y aplicarlo al modelo base

!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()
# r más alto = más capacidad para aprender, pero también más parámetros entrenables

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


### **ENTRENAR CON LORA**

Con el dataset y LoRA ya configurados, ejecutamos el entrenamiento. La pérdida (*loss*) que reporta el entrenador es nuestra métrica objetiva: debería bajar a medida que el modelo aprende los ejemplos.

In [16]:
# Configurar el entrenador (SFTTrainer) y ejecutar el fine-tuning

from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados",
    num_train_epochs=30,
    per_device_train_batch_size=5,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
print("Pérdida final:", resultado_entrenamiento.training_loss)

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '2.554', 'grad_norm': '1.737', 'learning_rate': '0.0002', 'entropy': '2.25', 'num_tokens': '229', 'mean_token_accuracy': '0.5', 'epoch': '1'}
{'loss': '2.512', 'grad_norm': '1.753', 'learning_rate': '0.0001933', 'entropy': '2.244', 'num_tokens': '458', 'mean_token_accuracy': '0.4955', 'epoch': '2'}
{'loss': '2.456', 'grad_norm': '1.926', 'learning_rate': '0.0001867', 'entropy': '2.234', 'num_tokens': '687', 'mean_token_accuracy': '0.5089', 'epoch': '3'}
{'loss': '2.387', 'grad_norm': '2.05', 'learning_rate': '0.00018', 'entropy': '2.223', 'num_tokens': '916', 'mean_token_accuracy': '0.5268', 'epoch': '4'}
{'loss': '2.317', 'grad_norm': '1.984', 'learning_rate': '0.0001733', 'entropy': '2.208', 'num_tokens': '1145', 'mean_token_accuracy': '0.5223', 'epoch': '5'}
{'loss': '2.246', 'grad_norm': '2.011', 'learning_rate': '0.0001667', 'entropy': '2.188', 'num_tokens': '1374', 'mean_token_accuracy': '0.5446', 'epoch': '6'}
{'loss': '2.17', 'grad_norm': '2.193', 'learning_rate': '0.0

### **DESPUÉS DEL FINE-TUNING: MEDIR LA MEJORA**

Compararemos la pérdida antes y después del entrenamiento como métrica objetiva.



In [17]:
# Comparar la pérdidas

perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = resultado_entrenamiento.training_loss

print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

# trainer.state.log_history[0]['loss'] es la pérdida después del primer paso registrado,
# no la pérdida real del modelo sin ningún entrenamiento.

Pérdida al inicio del entrenamiento: 2.55
Pérdida final del entrenamiento: 1.77
Reducción: 31%


In [20]:
# Referencia cualitativa vs Referencia con un modelo de producción

from groq import Groq


# 1. Modelo ajustado TinyLlama + LoRA
respuesta_lora = generar_respuesta(
    modelo_lora,
    prompt_prueba
)

# 2. Modelo de referencia
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt_evaluacion
        }
    ],
    temperature=0.2
)

respuesta_referencia = response.choices[0].message.content

# 3. Mostrar resultados
print("=" * 60)
print("MODELO AJUSTADO — TinyLlama + LoRA")
print("=" * 60)
print(respuesta_lora)

print("\n" + "=" * 60)
print("MODELO DE REFERENCIA — GPT-OSS-20B")
print("=" * 60)
print(respuesta_referencia)

MODELO AJUSTADO — TinyLlama + LoRA
Sleep, and the client.

MODELO DE REFERENCIA — GPT-OSS-20B
Sí, si el pedido aún no se ha enviado. Contáctanos lo antes posible.


**Nota:** Por eso, además de la respuesta de TinyLlama, se muestra una respuesta de Groq (GPT-OSS-20B) para el mismo prompt: no porque el proceso de fine-tuning haya fallado, sino como punto de comparación de cómo respondería un modelo de producción con muchos más parámetros y datos de entrenamiento — TinyLlama con 5 ejemplos demuestra la técnica de LoRA, no busca igualar la calidad de un modelo así de grande.